In [3]:
import os
import cv2
from numpy.random import f
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from collections import Counter
import albumentations as A

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
dataset_path = "Data"

shoplifters_path = os.path.join(dataset_path,"shop lifters")
nonshoplifters_path = os.path.join(dataset_path,"non shop lifters")

print("Shoplifters videos:", len(os.listdir(shoplifters_path)))
print("Non-shoplifters videos:", len(os.listdir(nonshoplifters_path)))

Shoplifters videos: 301
Non-shoplifters videos: 295


In [5]:
def video_hash(video_path, num_frames=20):

    cap = cv2.VideoCapture(video_path)

    hashes = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total_frames // num_frames, 1)

    for i in range(num_frames):

        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(frame)

        h = imagehash.phash(img)
        hashes.append(str(h))

    cap.release()

    return tuple(hashes)

In [6]:
video_hashes = {}
duplicates = []

for root,_,files in os.walk(dataset_path):

    for file in files:

        if file.endswith(".mp4"):

            path = os.path.join(root,file)

            h = video_hash(path)

            if h in video_hashes:
                duplicates.append((path,video_hashes[h]))
            else:
                video_hashes[h] = path

print("Number of duplicates:", len(duplicates))
duplicates[:5]

Number of duplicates: 0


[]

In [5]:
for dup in duplicates:
    os.remove(dup[0])

print("Duplicates removed")

Duplicates removed


In [7]:
def extract_frames(video_path, max_frames=32, img_size=96):

    cap = cv2.VideoCapture(video_path)
    frames = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total_frames // max_frames, 1)

    for i in range(max_frames):

        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame, (img_size, img_size))
        frames.append(frame)

    cap.release()

    frames = np.array(frames)

    if len(frames) < max_frames:
        padding = np.zeros((max_frames-len(frames), img_size, img_size, 3))
        frames = np.concatenate((frames, padding))

    return frames

In [8]:
transform = A.Compose([
    
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Rotate(limit=10,p=0.3)
])

def augment_video(frames):

    aug_frames = []

    for frame in frames:

        augmented = transform(image=frame)['image']
        aug_frames.append(augmented)

    return np.array(aug_frames)

In [9]:
videos = []
labels = []

classes = ["non shop lifters","shop lifters"]

for label,cls in enumerate(classes):

    folder = os.path.join(dataset_path,cls)

    for vid in tqdm(os.listdir(folder)):

        path = os.path.join(folder,vid)

        frames = extract_frames(path)

        videos.append(frames)
        labels.append(label)

X = np.array(videos)
y = np.array(labels)

print("Dataset shape:",X.shape)

100%|██████████| 301/301 [04:10<00:00,  1.20it/s]

Dataset shape: (596, 32, 96, 96, 3)


In [10]:
X = X / 255.0
X = np.transpose(X,(0,4,1,2,3))

Counter(y)

Counter({np.int64(1): 301, np.int64(0): 295})

In [11]:
class_counts = Counter(y)

total = sum(class_counts.values())

weights = [total/class_counts[i] for i in range(len(class_counts))]

class_weights = torch.tensor(weights,dtype=torch.float)

class_weights

tensor([2.0203, 1.9801])

In [12]:
X_train,X_test,y_train,y_test = train_test_split(
    
    X,y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [13]:
from torch.utils.data import Dataset, DataLoader

class VideoDataset(Dataset):

    def __init__(self,X,y):

        self.X = torch.tensor(X,dtype=torch.float32)
        self.y = torch.tensor(y,dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):

        return self.X[idx],self.y[idx]

In [14]:
train_dataset = VideoDataset(X_train,y_train)
test_dataset = VideoDataset(X_test,y_test)

train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=8)